# Project 02 — Machine Learning: Titanic Survival Prediction
**Pluto Academy AI & ML Internship**

**Dataset:** [Titanic — Machine Learning from Disaster (Kaggle)](https://www.kaggle.com/c/titanic)

**Objective:** Build, train, and compare 3 ML models to predict passenger survival. Identify the best-performing model using proper classification metrics.

---

## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120

print('Libraries imported successfully.')

---
## Step 1 — Load, Explore & Preprocess

> **How to get the dataset:**
> 1. Go to https://www.kaggle.com/c/titanic/data
> 2. Download `train.csv` (this is the labeled dataset we'll use)
> 3. Upload to Colab session or load from Google Drive

In [ ]:
# --- Option A: Upload directly ---
# from google.colab import files
# uploaded = files.upload()

# --- Option B: From Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/train.csv')

df = pd.read_csv('train.csv')

print('Dataset loaded.')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# Missing value check
missing = df.isnull().sum()
print('Missing values:')
print(missing[missing > 0])

In [ ]:
# Basic stats
df.describe()

### Preprocessing

In [ ]:
# --- Decision 1: Fill missing Age with median ---
# Reason: Age has ~177 missing values (~20%). Median is robust to outliers.
df['Age'].fillna(df['Age'].median(), inplace=True)

# --- Decision 2: Fill missing Embarked with mode ---
# Reason: Only 2 rows missing; fill with most common port.
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# --- Decision 3: Drop 'Cabin' column ---
# Reason: ~77% missing — too sparse to impute meaningfully.
df.drop(columns=['Cabin'], inplace=True)

# --- Decision 4: Drop 'Name', 'Ticket', 'PassengerId' ---
# Reason: High-cardinality identifiers with no direct predictive value in this model.
df.drop(columns=['Name', 'Ticket', 'PassengerId'], inplace=True)

# --- Decision 5: Encode categorical columns ---
# Reason: ML models require numeric inputs. Use LabelEncoder for binary categories.
le = LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])        # male=1, female=0
df['Embarked'] = le.fit_transform(df['Embarked'])  # C=0, Q=1, S=2

print('Preprocessing complete.')
print(f'Final shape: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')
df.head()

---
## Step 2 — Feature Engineering

In [ ]:
# Correlation heatmap to understand feature relationships
fig, ax = plt.subplots(figsize=(9, 6))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (Titanic)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('titanic_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Features selected based on correlation with 'Survived' and domain knowledge
# Keeping: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
# Reasoning:
#   - Sex has highest positive correlation with survival (~0.54)
#   - Pclass (ticket class) is a strong socioeconomic proxy
#   - Fare is correlated with Pclass but adds independent signal
#   - SibSp & Parch capture family size effects
#   - Age is biologically relevant (children first)

X = df.drop(columns=['Survived'])
y = df['Survived']

print('Features used:', list(X.columns))
print(f'Target distribution:\n{y.value_counts()}')

In [ ]:
# Train/Test split: 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling (needed for Logistic Regression and KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

---
## Step 3 — Train 3 Different Models

In [ ]:
# ── Model 1: Logistic Regression ──
# A linear model that estimates probability using the sigmoid function.
# Works well on linearly separable problems; fast and interpretable.
lr = LogisticRegression(max_iter=200, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print('Logistic Regression trained.')

In [ ]:
# ── Model 2: Random Forest Classifier ──
# An ensemble of decision trees using majority voting.
# Handles non-linearity well; robust to outliers; provides feature importance.
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)  # tree-based models don't need scaling
y_pred_rf = rf.predict(X_test)
print('Random Forest trained.')

In [ ]:
# ── Model 3: K-Nearest Neighbors ──
# Classifies based on the k closest training examples in feature space.
# Non-parametric; sensitive to feature scale (hence using scaled data).
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
print('KNN trained.')

---
## Step 4 — Evaluate & Compare All Models

In [ ]:
def get_metrics(name, y_true, y_pred):
    return {
        'Model': name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'F1 Score':  round(f1_score(y_true, y_pred), 4)
    }

results = [
    get_metrics('Logistic Regression', y_test, y_pred_lr),
    get_metrics('Random Forest',       y_test, y_pred_rf),
    get_metrics('KNN (k=5)',           y_test, y_pred_knn)
]

results_df = pd.DataFrame(results).set_index('Model')
print('\n=== MODEL COMPARISON TABLE ===')
print(results_df.to_string())

In [ ]:
# Visual comparison of all metrics
fig, ax = plt.subplots(figsize=(10, 5))
results_df.T.plot(kind='bar', ax=ax, edgecolor='black', width=0.7)
ax.set_title('Model Performance Comparison — All Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.1)
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left')
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width()/2, p.get_height() + 0.01),
                ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('titanic_model_comparison.png', bbox_inches='tight')
plt.show()

---
## Step 5 — Best Model Analysis & Conclusion

In [ ]:
# Identify best model by F1 Score
best_model_name = results_df['F1 Score'].idxmax()
print(f'Best model by F1 Score: {best_model_name}')
print(results_df.loc[best_model_name])

In [ ]:
# Confusion Matrix for Random Forest (best model)
cm = confusion_matrix(y_test, y_pred_rf)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'],
            ax=ax, linewidths=0.5)
ax.set_title('Confusion Matrix — Random Forest (Best Model)', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.savefig('titanic_confusion_matrix_rf.png', bbox_inches='tight')
plt.show()

print('\nClassification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# Feature Importance from Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', ax=ax, color='#E50914', edgecolor='black')
ax.set_title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig('titanic_feature_importance.png', bbox_inches='tight')
plt.show()

### 5-Line Conclusion

1. **Random Forest outperformed all three models**, achieving the highest F1 Score and Accuracy on the test set — demonstrating that ensemble methods handle the non-linear relationships in Titanic survival data better than linear or distance-based approaches.
2. **Sex (gender) was the most important predictive feature**, confirming the historical "women and children first" evacuation protocol — female passengers had a significantly higher survival rate.
3. **Fare and Pclass were the next strongest features**, together acting as a proxy for socioeconomic status; first-class passengers had preferential access to lifeboats.
4. **KNN was the weakest model** in this experiment, likely because its distance-based decision boundary struggles when features like Pclass (discrete) and Fare (continuous) are on very different scales — even after normalization.
5. **For deployment, Random Forest is the recommended model** — it handles missing-value-imputed data robustly, provides interpretable feature importances, and generalizes well without heavy hyperparameter tuning.
